## Imports

In [23]:
import os
from pyspark.sql import SparkSession

# os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Bigdata") \
    .config("spark.sql.repl.eagerEval.enabled", "true") \
    .getOrCreate()

print("Spark session started ✅")

Spark session started ✅


In [24]:
base_path = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
base_path = os.path.join(base_path, "data")

parquets = ["orders", "customers", "order_items", "payments", "products", "sellers", "reviews"]

dataframes = {}
i = 0
for parquet in parquets:
    df = spark.read.parquet(f"{os.path.join(base_path, "2_silver", parquet)}")
    dataframes[parquet] = df
    i += 1
print(f"{i} dataframes successfully loaded ✅")

7 dataframes successfully loaded ✅


## Gold tables

### Sales KPIs (total and monthly CA, average cart, payments distribution)

In [25]:
from pyspark.sql import functions as F

df_sales = dataframes["orders"].filter(F.col("order_status") != "canceled") \
    .join(dataframes["payments"], on="order_id", how="inner") \
    .withColumn("month_year", F.date_format("order_purchase_timestamp", "yyyy-MM")) \
    .select("order_id", "customer_id", "payment_value", "payment_type", "month_year")


### Delivery and reviews KPIs (delivery delay, lateness rate, average rate)

In [26]:
df_deliveries_reviews = dataframes["orders"].filter(F.col("order_status") == "delivered") \
    .withColumn("delay_days", F.datediff("order_delivered_customer_date", "order_purchase_timestamp")) \
    .withColumn("is_late", F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), True).otherwise(False)) \
    .select("order_id", "delay_days", "is_late") \
    .join(dataframes["reviews"], on="order_id", how="inner") \
    .select("order_id", "delay_days", "is_late", "review_score")


### Tops KPIs (top sellers, top categories)

In [27]:
df_products_sellers = dataframes["order_items"] \
    .join(dataframes["products"], on="product_id", how="inner") \
    .join(dataframes["sellers"], on="seller_id", how="inner") \
    .select("order_id", "product_id", "product_category_name_english", "price", "seller_id", "seller_city", "seller_state")

### Geo customers KPIs (to be joined to df_sales to get CA by city or state)

In [28]:
df_geo_customers = dataframes["orders"] \
    .join(dataframes["customers"], on="customer_id", how="inner") \
    .select("order_id", "customer_id", "customer_zip_code_prefix", "customer_city", "customer_state")

## Exports

In [29]:
gold_dataframes = {
    "sales": df_sales,
    "deliveries_reviews": df_deliveries_reviews,
    "products_sellers": df_products_sellers,
    "geo_customers": df_geo_customers
}

for name, df in gold_dataframes.items():
    df.write.mode("overwrite").parquet(f"{base_path}/3_gold/{name}")
    print(f"Dataframe {name} saved to gold ✅")

Dataframe sales saved to gold ✅


Dataframe deliveries_reviews saved to gold ✅


Dataframe products_sellers saved to gold ✅


Dataframe geo_customers saved to gold ✅
